# Week6 복습과제 — YOLOv8 포즈 추정 실습
교재 9장 객체 탐지 · 모델 실습: YOLOv8 (p.591–600) 예제 9.25 ~ 9.30

### 울트라리틱스 라이브러리 설치

In [1]:
"""
터미널 가상환경에서 실행
`!pip install ultralytics opencv-python`
"""

'\n터미널 가상환경에서 실행\n`!pip install ultralytics opencv-python`\n'

### 예제 9.25 YOLOv8 포즈 추정 모델 불러오기

In [2]:
from ultralytics import YOLO

# YOLO(You Only Look Once): 이미지 전체를 한 번만 보고 경계 상자 위치와 클래스를 동시에 예측하는 1-stage 탐지기
# YOLOv8: 앵커 프리(anchor-free) 구조 → 미리 정의한 앵커 박스 없이 객체의 위치를 직접 예측
# 모델 크기: n < s < m < l < x (클수록 정확하지만 느림)
# 접미사 -pose: 포즈 추정 모델 (COCO 'person' 클래스 탐지 + 사람마다 키포인트 17개 예측)
model = YOLO("../models/yolov8m-pose.pt")

### 예제 9.26 비디오 파일 불러오기
> Google Colab에서는 `cv2.imshow` 대신 `from google.colab.patches import cv2_imshow` 후 `cv2_imshow(frame)`을 사용

In [3]:
import cv2

# YOLO는 이미지 단위로 추론하므로, 비디오를 프레임 단위로 읽어 모델에 입력한다
capture = cv2.VideoCapture("../datasets/woman.mp4")
while cv2.waitKey(10) < 0:
    if capture.get(cv2.CAP_PROP_POS_FRAMES) == capture.get(cv2.CAP_PROP_FRAME_COUNT):
        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)

    ret, frame = capture.read()
    cv2.imshow("VideoFrame", frame)

capture.release()
cv2.destroyAllWindows()

### 예제 9.27 모델 추론

In [4]:
import torch

def predict(frame, iou=0.7, conf=0.25):
    results = model(
        source=frame,
        device="0" if torch.cuda.is_available() else "cpu",
        iou=0.7,  # NMS IoU 임계값: 서로 IoU가 이 값 이상 겹치는 상자 중 점수가 낮은 쪽을 중복으로 보고 제거
        conf=0.25,  # 신뢰도 임계값: 점수가 이보다 낮은 예측은 버림
        verbose=False,
    )
    result = results[0]  # 결과는 배치(이미지 N장) 단위 리스트 → 프레임 1장이므로 첫 번째만 사용
    return result

### 예제 9.28 경계 상자 시각화

In [5]:
def draw_boxes(result, frame):
    for boxes in result.boxes:
        # boxes.data: [x1, y1, x2, y2, conf, cls] = 좌상단·우하단 좌표, 신뢰도, 클래스 인덱스
        x1, y1, x2, y2, score, classes = boxes.data.squeeze().cpu().numpy()
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 1)
    return frame

### 예제 9.29 모델 추론 및 시각화 적용

In [6]:
capture = cv2.VideoCapture("../datasets/woman.mp4")
while cv2.waitKey(10) < 0:
    if capture.get(cv2.CAP_PROP_POS_FRAMES) == capture.get(cv2.CAP_PROP_FRAME_COUNT):
        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)

    ret, frame = capture.read()
    result = predict(frame)  # 프레임마다 YOLO 추론 (NMS까지 적용된 최종 결과)
    frame = draw_boxes(result, frame)
    cv2.imshow("VideoFrame", frame)

capture.release()
cv2.destroyAllWindows()

### 예제 9.30 키 포인트 시각화

In [12]:
from ultralytics.utils.plotting import Annotator

def draw_keypoints(result, frame):
    annotator = Annotator(frame, line_width=1)
    for kps in result.keypoints:  # 탐지된 사람(경계 상자) 하나당 키포인트 한 세트
        kps = kps.data.squeeze()  # [17, 3] = 17개 관절의 (x, y, conf)
        annotator.kpts(kps)  # conf 0.5 이상인 키포인트만 점과 골격 선으로 그림

        nkps = kps.cpu().numpy()
        # nkps[:,2] = 1
        # annotator.kpts(nkps)
        for idx, (x, y, score) in enumerate(nkps):
            if score > 0.5:
                cv2.circle(frame, (int(x), int(y)), 3, (0, 0, 255), cv2.FILLED)
                cv2.putText(
                    frame, str(idx), (int(x), int(y)), cv2.FONT_HERSHEY_COMPLEX,
                    1, (0, 0, 255), 1
                )

    return frame

### 키 포인트 시각화 적용

In [13]:
capture = cv2.VideoCapture("../datasets/woman.mp4")
while cv2.waitKey(10) < 0:
    if capture.get(cv2.CAP_PROP_POS_FRAMES) == capture.get(cv2.CAP_PROP_FRAME_COUNT):
        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)

    ret, frame = capture.read()
    result = predict(frame)  # 프레임마다 YOLO 추론 (NMS까지 적용된 최종 결과)
    frame = draw_boxes(result, frame)
    frame = draw_keypoints(result, frame)  # 경계 상자(탐지) + 키포인트(포즈)를 함께 시각화
    cv2.imshow("VideoFrame", frame)

capture.release()
cv2.destroyAllWindows()